# 📊 Notebook 01: Decomposição de Séries Temporais — Grupo B (SC)
**Disciplina**: Técnicas de IA Aplicadas a Sistemas de Energia  
**Programa**: Mestrado Profissional — IFSC  
**Autor**: Dilson Eijo Rigotti  

---

## 🎯 Objetivo
Analisar a decomposição estatística (Tendência, Sazonalidade e Resíduos) das séries temporais de consumo elétrico (MWh) das classes de baixa tensão (**Residencial, Comercial e Rural**) do Estado de Santa Catarina, preparando os dados para os modelos preditivos (SARIMAX e LSTM).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import json

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Carregamento dos Dados Processados

In [ ]:
df = pd.read_csv('../data/processed/sc_grupo_b_mensal.csv')
df['data'] = pd.to_datetime(df['data'])
df.head()

## 2. Decomposição STL (Seasonal-Trend decomposition using LOESS)
A técnica STL decompõe a série temporal $Y_t$ em três componentes aditivos:
$$Y_t = T_t + S_t + R_t$$
onde $T_t$ é a tendência de longo prazo, $S_t$ é o padrão sazonal anual de 12 meses, e $R_t$ é o resíduo/ruído.

In [ ]:
classes = ['Residencial', 'Comercial', 'Rural']

for c in classes:
    df_c = df[df['classe'] == c].sort_values('data').copy()
    df_c.set_index('data', inplace=True)
    df_c = df_c.asfreq('MS')
    
    ts_consumo = df_c['consumo_mwh'].interpolate(method='linear')
    
    stl = STL(ts_consumo, seasonal=13, robust=True)
    res = stl.fit()
    
    fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
    axes[0].plot(res.observed, color='#1f77b4', linewidth=1.5)
    axes[0].set_ylabel('Observado (MWh)')
    axes[0].set_title(f'Decomposição STL - Classe {c} (SC 1994-2026)', fontsize=13, fontweight='bold')
    
    axes[1].plot(res.trend, color='#ff7f0e', linewidth=2)
    axes[1].set_ylabel('Tendência')
    
    axes[2].plot(res.seasonal, color='#2ca02c', linewidth=1.5)
    axes[2].set_ylabel('Sazonalidade')
    
    axes[3].plot(res.resid, color='#d62728', marker='.', linestyle='None', alpha=0.5)
    axes[3].axhline(0, color='black', linestyle='--', linewidth=0.8)
    axes[3].set_ylabel('Resíduo')
    axes[3].set_xlabel('Ano')
    
    plt.tight_layout()
    plt.show()

## 3. Autocorrelação (ACF) e Autocorrelação Parcial (PACF)
A análise de ACF e PACF auxilia na identificação dos parâmetros de ordem $(p, d, q) \times (P, D, Q)_{12}$ para a modelagem SARIMAX.

In [ ]:
for c in classes:
    df_c = df[df['classe'] == c].sort_values('data').copy()
    ts_consumo = df_c.set_index('data')['consumo_mwh'].asfreq('MS').interpolate(method='linear')
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    plot_acf(ts_consumo, ax=ax1, lags=36, title=f'Autocorrelação (ACF) - {c}')
    plot_pacf(ts_consumo, ax=ax2, lags=36, title=f'Autocorrelação Parcial (PACF) - {c}')
    plt.tight_layout()
    plt.show()

## 4. Métricas de Força da Sazonalidade e Tendência

In [ ]:
with open('../data/processed/stl_metrics_summary.json', 'r', encoding='utf-8') as f:
    summary = json.load(f)

df_metrics = pd.DataFrame(summary).T
df_metrics